In [ ]:
import subprocess, sys

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--index-url", "https://download.pytorch.org/whl/cu121",
     "torch==2.4.0", "torchvision==0.19.0"],
    capture_output=True, text=True
)
print("torch:", "OK" if r.returncode == 0 else r.stderr[-300:])

r2 = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.47.1", "accelerate>=0.26.0", "bitsandbytes>=0.42.0",
     "sentence-transformers==3.0.1", "pymupdf==1.25.5", "faiss-cpu",
     "gradio==4.44.1", "qwen-vl-utils", "Pillow", "tqdm", "pandas","colpali-engine>=0.3.1"],
    capture_output=True, text=True
)
print("Other:", "OK" if r2.returncode == 0 else r2.stderr[-400:])


torch: OK
Other: OK


In [ ]:
import torch
print(f"CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA : True
GPU  : Tesla T4
VRAM : 15.6 GB


In [ ]:
import os
os.makedirs("/content/src", exist_ok=True)
os.makedirs("/content/data/pdfs", exist_ok=True)


In [ ]:
%%writefile /content/src/ingestion.py

import os
import re
import fitz
import pandas as pd
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from dataclasses import dataclass, field
from typing import List, Dict, Any, Tuple

MAX_PAGES = 20
MAX_CAPTIONS_PER_PAGE = 2
CHUNK_MAX_TOKENS = 300
CHUNK_OVERLAP_PARAS= 1

@dataclass
class DocumentChunk:
    chunk_id:   str
    page_num:   int
    chunk_type: str
    content:    str
    source_doc: str
    metadata:   Dict[str, Any] = field(default_factory=dict)


class MultiModalIngestionPipeline:
    CAPTION_RE = re.compile(
        r"^(Figure|Fig\.|Chart|Graph|Exhibit|Box)\s*[\dIVX\.]+",
        re.IGNORECASE,
    )

    HEADING_PATTERNS = [
        re.compile(r"^[IVX]+\.\s+[A-Z][A-Za-z\s]{2,50}$"),
        re.compile(r"^\d+\.\s+[A-Z][A-Z\s]{4,60}$"),
        re.compile(r"^[A-Z][A-Z\s]{6,60}$"),
        re.compile(r"^(Box|Figure|Table|Annex|Appendix)\s+[\dIVX]"),
        re.compile(r"^(Chapter|CHAPTER|SECTION|Section)\s+\d+"),
        re.compile(r"^[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3}$"),
    ]

    def __init__(self, dpi: int = 150):
        self.dpi = dpi

    def _render_page_image(self, page) -> Image.Image:
        mat = fitz.Matrix(self.dpi / 72, self.dpi / 72)
        pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
        return Image.open(BytesIO(pix.tobytes("png"))).convert("RGB")

    def _is_heading(self, text: str) -> bool:
        t = text.strip()
        if not t or len(t) > 100:
            return False
        if t.endswith(".") and len(t.split()) > 4:
            return False
        if len(t.split()) > 10:
            return False
        return any(p.match(t) for p in self.HEADING_PATTERNS)

    def _extract_tables(self, page, page_num: int, source_doc: str) -> List[DocumentChunk]:
        chunks = []
        try:
            for t_idx, table in enumerate(page.find_tables()):
                raw = table.extract()
                if not raw or len(raw) < 2:
                    continue
                df = pd.DataFrame(raw)
                first_row = [str(c).strip() for c in df.iloc[0].fillna("")]
                if any(first_row):
                    df.columns = [h or f"Col{i}" for i, h in enumerate(first_row)]
                    df = df.iloc[1:].reset_index(drop=True)
                df = df.fillna("")
                if df.shape[0] < 1 or df.shape[1] < 2:
                    continue
                table_text = (
                    f"[TABLE | {source_doc} | Page {page_num}]\n"
                    + df.to_string(index=False)
                )
                chunks.append(DocumentChunk(
                    chunk_id=f"{source_doc}_table_p{page_num}_t{t_idx}",
                    page_num=page_num,
                    chunk_type="table",
                    content=table_text,
                    source_doc=source_doc,
                    metadata={"rows": df.shape[0], "cols": df.shape[1]},
                ))
        except Exception as exc:
            print(f"  [table warn] p{page_num}: {exc}")
        return chunks

    def _extract_image_captions(self, page, page_num: int, source_doc: str) -> List[DocumentChunk]:
        seen_labels: set = set()
        raw_candidates: List[DocumentChunk] = []
        try:
            text  = page.get_text("text")
            lines = [l.strip() for l in text.split("\n") if l.strip()]
            for i, line in enumerate(lines):
                if not self.CAPTION_RE.match(line):
                    continue
                label_key = re.sub(r"\s+", " ", line[:50]).lower().strip()
                if label_key in seen_labels:
                    continue
                seen_labels.add(label_key)
                caption_lines = [line] + lines[i + 1: i + 5]
                caption_text  = (
                    f"[FIGURE/CHART | {source_doc} | Page {page_num}]\n"
                    + " ".join(caption_lines)
                )
                raw_candidates.append(DocumentChunk(
                    chunk_id=f"{source_doc}_fig_p{page_num}_l{i}",
                    page_num=page_num,
                    chunk_type="image_caption",
                    content=caption_text,
                    source_doc=source_doc,
                    metadata={"caption_label": line},
                ))
        except Exception:
            pass
        raw_candidates.sort(key=lambda c: len(c.content), reverse=True)
        return raw_candidates[:MAX_CAPTIONS_PER_PAGE]

    def _chunk_text(
        self,
        text: str,
        page_num: int,
        section: str,
        source_doc: str,
        max_tokens: int = CHUNK_MAX_TOKENS,
    ) -> List[DocumentChunk]:

        paragraphs = [p.strip() for p in re.split(r"\n{2,}", text.strip()) if p.strip()]
        if not paragraphs:
            return []

        chunks: List[DocumentChunk] = []
        chunk_idx      = 0
        i              = 0
        last_chunk_end = 0

        while i < len(paragraphs):
            current_paras: List[str] = []
            cur_len = 0

            if chunk_idx > 0 and last_chunk_end > 0:
                overlap_start = max(0, last_chunk_end - CHUNK_OVERLAP_PARAS)
                for op in paragraphs[overlap_start:last_chunk_end]:
                    current_paras.append(op)
                    cur_len += len(op.split())

            chunk_start = i
            while i < len(paragraphs):
                para = paragraphs[i]
                plen = len(para.split())
                if cur_len + plen > max_tokens and current_paras:
                    break
                current_paras.append(para)
                cur_len += plen
                i += 1

            last_chunk_end = i

            combined = " ".join(current_paras).strip()
            if len(combined) < 40:
                chunk_idx += 1
                continue

            chunks.append(DocumentChunk(
                chunk_id=f"{source_doc}_text_p{page_num}_c{chunk_idx}",
                page_num=page_num,
                chunk_type="heading" if self._is_heading(combined) else "text",
                content=combined,
                source_doc=source_doc,
                metadata={"section": section},
            ))
            chunk_idx += 1

        return chunks

    def ingest(self, pdf_paths: List[str]) -> Tuple[List[DocumentChunk], list]:
        all_chunks: List[DocumentChunk] = []
        page_images: list = []

        for pdf_path in pdf_paths:
            source_doc       = os.path.splitext(os.path.basename(pdf_path))[0]
            doc              = fitz.open(pdf_path)
            total_pages      = len(doc)
            pages_to_process = min(total_pages, MAX_PAGES)
            print(
                f"\n {source_doc}: {total_pages} pages total — "
                f"processing first {pages_to_process}"
            )

            section = "Introduction"
            for page_num in tqdm(range(pages_to_process), desc=f"  {source_doc}"):
                page = doc[page_num]
                pnum = page_num + 1

                page_images.append((pnum, source_doc, self._render_page_image(page)))

                text = page.get_text("text")

                for line in text.split("\n"):
                    stripped = line.strip()
                    if stripped and self._is_heading(stripped):
                        section = stripped[:80]
                        break

                all_chunks.extend(self._extract_tables(page, pnum, source_doc))
                all_chunks.extend(self._extract_image_captions(page, pnum, source_doc))
                all_chunks.extend(self._chunk_text(text, pnum, section, source_doc))

            doc.close()

        type_counts: Dict[str, int] = {}
        for c in all_chunks:
            type_counts[c.chunk_type] = type_counts.get(c.chunk_type, 0) + 1

        print(f"\n Ingestion complete — {len(all_chunks)} chunks from {len(pdf_paths)} PDFs")
        for t, n in sorted(type_counts.items()):
            print(f"   {t}: {n}")
        print(f"   Page images: {len(page_images)}")
        return all_chunks, page_images

Overwriting /content/src/ingestion.py


In [ ]:
%%writefile /content/src/indexing.py

import torch
import numpy as np
import faiss
from tqdm import tqdm
from typing import List, Dict
from sentence_transformers import SentenceTransformer
from transformers import ColPaliForRetrieval, ColPaliProcessor, BitsAndBytesConfig


class ColPaliIndexer:
    def __init__(self, model_name: str = "vidore/colpali-v1.3-hf"):
        print(f"Loading ColPali: {model_name}")
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        self.model = ColPaliForRetrieval.from_pretrained(
            model_name,
            quantization_config=bnb,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        ).eval()
        self.processor = ColPaliProcessor.from_pretrained(model_name)
        self.page_embeddings: List[Dict] = []
        print("ColPali loaded (4-bit quantized)")

    def _to_device(self, inputs: dict) -> dict:
        return {
            k: v.to(self.model.device).to(torch.bfloat16)
            if torch.is_floating_point(v) else v.to(self.model.device)
            for k, v in inputs.items()
        }

    @torch.no_grad()
    def embed_pages(self, page_images: list, batch_size: int = 2):

        print(f"\n Embedding {len(page_images)} pages with ColPali...")
        self.page_embeddings = []

        for i in tqdm(range(0, len(page_images), batch_size), desc="ColPali pages"):
            batch  = page_images[i: i + batch_size]
            images = [b[2] for b in batch]
            try:

                if hasattr(self.processor, "process_images"):
                    inputs = self.processor.process_images(images)
                else:
                    inputs = self.processor(images=images, return_tensors="pt")

                inputs = self._to_device(inputs)
                outputs = self.model(**inputs)

                for idx, emb in enumerate(outputs.embeddings):
                    pnum, sdoc, _ = batch[idx]
                    self.page_embeddings.append({
                        "page_num":   pnum,
                        "source_doc": sdoc,
                        "embedding":  emb.detach().cpu().float(),
                    })
            except Exception as exc:
                print(f"  [ColPali page warn] batch {i}: {exc}")

        print(f" {len(self.page_embeddings)} page embeddings stored")

    @torch.no_grad()
    def query(self, query_text: str, top_k: int = 5) -> List[Dict]:

        if not self.page_embeddings:
            return []
        try:
            if hasattr(self.processor, "process_queries"):
                inputs = self.processor.process_queries([query_text])
            else:
                inputs = self.processor(text=[query_text], return_tensors="pt")

            inputs = self._to_device(inputs)
            q_emb  = self.model(**inputs).embeddings[0].detach().cpu().float()
        except Exception as exc:
            print(f"  [ColPali query warn]: {exc}")
            return []

        scores = []
        for pd_ in self.page_embeddings:
          sim = torch.matmul(q_emb, pd_["embedding"].T)
          score = sim.max(dim=1).values.sum().item()
          scores.append((pd_["page_num"], pd_["source_doc"], score))
        scores.sort(key=lambda x: x[2], reverse=True)
        return [
            {"page_num": p, "source_doc": s, "score": sc}
            for p, s, sc in scores[:top_k]
        ]


class TextIndexer:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print(f"Loading text embedder: {model_name}")
        self.model  = SentenceTransformer(model_name)
        self.dim    = self.model.get_sentence_embedding_dimension()
        self.index  = None
        self.chunks = []
        print(f"Text embedder loaded (dim={self.dim})")

    def build_index(self, chunks, batch_size: int = 64):
        self.chunks = list(chunks)
        texts = [c.content for c in self.chunks]
        print(f"\n Embedding {len(texts)} chunks with MiniLM...")
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        ).astype(np.float32)

        n = len(self.chunks)
        if n > 1000:
            nlist     = max(4, min(100, int(n ** 0.5)))
            quantizer = faiss.IndexFlatIP(self.dim)
            self.index = faiss.IndexIVFFlat(
                quantizer, self.dim, nlist, faiss.METRIC_INNER_PRODUCT
            )
            self.index.train(embeddings)
            self.index.nprobe = max(1, nlist // 4)
        else:
            self.index = faiss.IndexFlatIP(self.dim)

        self.index.add(embeddings)
        print(f"FAISS index built: {self.index.ntotal} vectors")

    def query(self, query_text: str, top_k: int = 5) -> List[Dict]:
        if self.index is None or not self.chunks:
            return []
        q_emb    = self.model.encode([query_text], normalize_embeddings=True).astype(np.float32)
        k_actual = min(top_k, len(self.chunks))
        scores, indices = self.index.search(q_emb, k_actual)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0 or idx >= len(self.chunks):
                continue
            c = self.chunks[idx]
            results.append({
                "chunk":      c,
                "score":      float(score),
                "page_num":   c.page_num,
                "chunk_type": c.chunk_type,
                "source_doc": c.source_doc,
            })
        return results

Overwriting /content/src/indexing.py


In [ ]:
%%writefile /content/src/retrieval.py

import re
from typing import List, Dict, Tuple, Set, Optional

CHUNK_TYPE_BUDGET = {
    "text": 4,
    "heading": 2,
    "table": 4,
    "image_caption": 5,
}

TABLE_QUERY_SIGNALS = re.compile(
    r"\b(percent|%|share|ratio|rate|gdp|growth|figure|table|how much|"
    r"how many|total|amount|number|population|billion|million|"
    r"classification|income|low.income|high.income|upper|lower|"
    r"extreme poverty|emissions|co2|projection|forecast|outlook)\b",
    re.IGNORECASE,
)

VISUAL_QUERY_SIGNALS = re.compile(
    r"\b(figure|fig\.|chart|graph|caption|box|image|diagram|"
    r"show|illustrate|depict|visual|plot)\b",
    re.IGNORECASE,
)

TABLE_PAGE_BONUS= 0.35
DOC_TARGET_BONUS = 0.20
TEXT_WEIGHT_DEFAULT = 1
VISUAL_WEIGHT_DEFAULT = 2
TEXT_WEIGHT_VISUAL = 1
VISUAL_WEIGHT_VISUAL  = 3

DOC_TARGET_PATTERNS: List[Tuple] = [
    (re.compile(r"\b(imf|weo|world economic outlook)\b", re.IGNORECASE), "imf_weo"),
    (re.compile(r"\b(world bank|wb report|middle.income trap)\b", re.IGNORECASE), "world_bank"),
]


class DualModeRetriever:
    def __init__(self, colpali, text_indexer, chunks, page_images):
        self.colpali      = colpali
        self.text_indexer = text_indexer
        self.chunks       = list(chunks)

        self.page_image_map: Dict[Tuple, object] = {
            (p[0], p[1]): p[2] for p in page_images
        }

        self._table_pages: Set[Tuple] = {
            (c.page_num, c.source_doc)
            for c in self.chunks if c.chunk_type == "table"
        }

        self._chunks_by_page: Dict[Tuple, List] = {}
        for c in self.chunks:
            key = (c.page_num, c.source_doc)
            self._chunks_by_page.setdefault(key, []).append(c)

    def _is_table_query(self, query: str) -> bool:
        return bool(TABLE_QUERY_SIGNALS.search(query))

    def _is_visual_query(self, query: str) -> bool:
        return bool(VISUAL_QUERY_SIGNALS.search(query))

    def _detect_target_doc(self, query: str) -> Optional[str]:
        for pattern, doc_key in DOC_TARGET_PATTERNS:
            if pattern.search(query):
                return doc_key
        return None

    def _rrf(self, rank_lists: List[List], weights: List[int], k: int = 60) -> Dict:
        scores: Dict = {}
        for rank_list, w in zip(rank_lists, weights):
            for rank, key in enumerate(rank_list):
                scores[key] = scores.get(key, 0.0) + w * (1.0 / (k + rank + 1))
        return scores

    def _apply_table_bonus(self, rrf_scores: Dict, query: str) -> Dict:
        if not self._is_table_query(query):
            return rrf_scores
        boosted = dict(rrf_scores)
        for key in boosted:
            if key in self._table_pages:
                boosted[key] += TABLE_PAGE_BONUS
        return boosted

    def _apply_doc_target_bonus(self, rrf_scores: Dict, target_doc: Optional[str]) -> Dict:
        if target_doc is None:
            return rrf_scores
        boosted = dict(rrf_scores)
        for (page_num, source_doc) in boosted:
            if target_doc in source_doc:
                boosted[(page_num, source_doc)] += DOC_TARGET_BONUS
        return boosted

    def _apply_type_budget(self, chunks: List, rank_order: List[Tuple]) -> List:
        rank_map = {key: i for i, key in enumerate(rank_order)}
        sorted_chunks = sorted(
            chunks,
            key=lambda c: (rank_map.get((c.page_num, c.source_doc), 999), c.page_num),
        )
        counts: Dict[str, int] = {}
        kept = []
        for c in sorted_chunks:
            budget = CHUNK_TYPE_BUDGET.get(c.chunk_type, 4)
            used   = counts.get(c.chunk_type, 0)
            if used < budget:
                kept.append(c)
                counts[c.chunk_type] = used + 1
        return kept

    def retrieve(self, query: str, top_k: int = 5) -> Dict:
        is_visual = self._is_visual_query(query)

        if is_visual:
            tw, vw = TEXT_WEIGHT_VISUAL, VISUAL_WEIGHT_VISUAL
        else:
            tw, vw = TEXT_WEIGHT_DEFAULT, VISUAL_WEIGHT_DEFAULT

        text_results   = self.text_indexer.query(query, top_k=top_k * 2)
        visual_results = self.colpali.query(query,      top_k=top_k * 2)

        seen: Set = set()
        text_keys: List[Tuple] = []
        for r in text_results:
            key = (r["page_num"], r["source_doc"])
            if key not in seen:
                seen.add(key)
                text_keys.append(key)

        seen_v: Set = set()
        visual_keys: List[Tuple] = []
        for r in visual_results:
            key = (r["page_num"], r["source_doc"])
            if key not in seen_v:
                seen_v.add(key)
                visual_keys.append(key)

        rrf_scores = self._rrf([text_keys, visual_keys], weights=[tw, vw])
        rrf_scores = self._apply_table_bonus(rrf_scores, query)
        target_doc = self._detect_target_doc(query)
        rrf_scores = self._apply_doc_target_bonus(rrf_scores, target_doc)

        top_keys = sorted(rrf_scores, key=lambda k: rrf_scores[k], reverse=True)[:top_k]

        candidate_chunks = []
        for key in top_keys:
            candidate_chunks.extend(self._chunks_by_page.get(key, []))

        retrieved_chunks = self._apply_type_budget(candidate_chunks, top_keys)

        context_parts = []
        for chunk in retrieved_chunks:
            header = (
                f"[{chunk.source_doc} | Page {chunk.page_num} "
                f"| {chunk.chunk_type.upper()}]"
            )
            if chunk.metadata.get("section"):
                header += f" — {chunk.metadata['section'][:60]}"
            context_parts.append(f"{header}\n{chunk.content}")

        context = "\n\n---\n\n".join(context_parts)

        if is_visual and visual_keys:
            visual_page_keys = visual_keys[:3]
        else:
            visual_page_keys = top_keys[:3]

        visual_pages = []
        for key in visual_page_keys:
            if key in self.page_image_map:
                visual_pages.append((key[0], key[1], self.page_image_map[key]))

        modality_counts: Dict[str, int] = {}
        for c in retrieved_chunks:
            modality_counts[c.chunk_type] = modality_counts.get(c.chunk_type, 0) + 1

        return {
            "context":          context,
            "source_pages":     top_keys,
            "retrieved_chunks": retrieved_chunks,
            "visual_pages":     visual_pages,
            "modality_counts":  modality_counts,
        }

Overwriting /content/src/retrieval.py


In [ ]:
%%writefile /content/src/generation.py

import re
import torch
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info

MAX_CONTEXT_CHARS = 4000
MAX_NEW_TOKENS    = 450


class AnswerGenerator:
    def __init__(self, model_name: str = "Qwen/Qwen2-VL-2B-Instruct"):
        print(f"Loading VLM: {model_name}")
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_name,
            quantization_config=bnb,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        ).eval()
        print(f" Qwen2-VL-2B loaded (4-bit, max_new_tokens={MAX_NEW_TOKENS})")

    def _build_page_ref(self, source_pages: list) -> str:
        if not source_pages:
            return "unknown source"
        parts = []
        for item in source_pages[:4]:
            if isinstance(item, tuple):
                page_num, source_doc = item
            else:
                page_num, source_doc = item, "doc"
            parts.append(f"p.{page_num} ({source_doc})")
        return ", ".join(parts)

    @staticmethod
    def _detect_target_doc(query: str):
        if re.search(r"\b(imf|weo|world economic outlook)\b", query, re.IGNORECASE):
            return "imf_weo"
        if re.search(r"\b(world bank)\b", query, re.IGNORECASE):
            return "world_bank"
        return None

    @staticmethod
    def _smart_trim_context(context: str, max_chars: int) -> str:

        if len(context) <= max_chars:
            return context

        sections = context.split("\n\n---\n\n")
        kept = []
        total = 0
        for sec in sections:
            sec_len = len(sec)
            if total + sec_len + 7 <= max_chars:
                kept.append(sec)
                total += sec_len + 7
            else:
                remaining = max_chars - total - 7
                if remaining > 200:
                    kept.append(sec[:remaining] + "\n[...trimmed...]")
                break
        return "\n\n---\n\n".join(kept)

    @staticmethod
    def _extract_table_highlights(context: str, query: str, target_doc=None) -> str:
        query_words = set(re.findall(r"\w+", query.lower()))
        stop = {"what", "the", "in", "for", "are", "is", "a", "an", "of",
                "and", "to", "do", "does", "shown", "show", "figure", "table",
                "imf", "weo", "world", "bank", "report", "appear", "appears"}
        query_words -= stop
        if not query_words:
            return ""

        highlights = []
        for section in re.split(r"\n\n---\n\n", context):
            if not re.search(r"\|\s*TABLE\s*\]", section, re.IGNORECASE):
                continue
            if target_doc and target_doc not in section.lower():
                continue
            lines = [l.strip() for l in section.splitlines() if l.strip()]
            if len(lines) < 2:
                continue
            header_line = lines[0]
            data_lines  = lines[1:]
            col_header  = data_lines[0] if data_lines else ""
            matched_rows = [
                row for row in data_lines[1:]
                if any(w in row.lower() for w in query_words)
            ]
            if matched_rows:
                highlights.append(
                    f"{header_line}\n{col_header}\n" + "\n".join(matched_rows[:8])
                )

        return ("RELEVANT TABLE DATA:\n" + "\n\n".join(highlights) + "\n\n") if highlights else ""

    @staticmethod
    def _extract_caption_highlights(context: str, query: str, target_doc=None) -> str:
        highlights = []
        for section in re.split(r"\n\n---\n\n", context):
            if not re.search(r"\|\s*IMAGE_CAPTION\s*\]", section, re.IGNORECASE):
                continue
            if target_doc and target_doc not in section.lower():
                continue
            lines = [l.strip() for l in section.splitlines() if l.strip()]
            if lines:
                caption_text = " ".join(lines[1:]) if len(lines) > 1 else lines[0]
                highlights.append(caption_text)
        return ("FIGURE/CHART CAPTIONS FOUND:\n" + "\n".join(highlights) + "\n\n") if highlights else ""

    def _build_prompt(self, query: str, context: str, page_ref: str) -> str:
        ctx = self._smart_trim_context(context, MAX_CONTEXT_CHARS)
        ctx = re.sub(r"\n{3,}", "\n\n", ctx).strip()

        target_doc = self._detect_target_doc(query)

        caption_highlight = ""
        if re.search(r"\b(caption|figure|chart|box|fig\.?)\b", query, re.IGNORECASE):
            caption_highlight = self._extract_caption_highlights(context, query, target_doc)

        table_highlight = ""
        if re.search(
            r"\b(rate|percent|%|growth|inflation|gdp|projection|output|"
            r"figure|table|number|amount)\b",
            query, re.IGNORECASE
        ):
            table_highlight = self._extract_table_highlights(context, query, target_doc)

        injected = caption_highlight + table_highlight

        prompt = (
    "You are a precise document QA assistant.\n"
    "Answer the QUESTION using ONLY the DOCUMENT EXCERPTS below.\n\n"

    "Rules:\n"
    "1. Use ONLY the provided excerpts.\n"
    "2. If a table is present, extract EXACT values from it.\n"
    "3. Do NOT summarize tables — copy the relevant numbers.\n"
    "4. If numbers are missing, say 'Not found'.\n"
    "5. Answer in 1-2 precise sentences.\n"
    f"6. End with exactly: [Source: {page_ref}]\n"
    f"7. If the answer is not found write: "
    f"'Not found in provided context. [Source: {page_ref}]'\n\n"

    f"{injected}"
    f"DOCUMENT EXCERPTS:\n{ctx}\n\n"
    f"QUESTION: {query}\n\n"
    "ANSWER:"
)
        return prompt

    @staticmethod
    def _strip_prompt_echo(answer: str) -> str:
        if "ANSWER:" in answer:
            answer = answer.split("ANSWER:")[-1]
        if "Rules:" in answer:
            answer = answer.split("Rules:")[-1]
        return answer.strip()

    @staticmethod
    def _clean_answer(answer: str) -> str:
        answer = re.sub(r"```.*?```", "", answer, flags=re.DOTALL)
        answer = re.sub(r"`", "", answer)
        answer = re.sub(r"\n{2,}", " ", answer)
        answer = re.sub(r"  +", " ", answer)
        return answer.strip()

    @torch.no_grad()
    def generate(
        self,
        query: str,
        context: str,
        visual_pages: list = None,
        source_pages: list = None,
    ) -> str:
        torch.cuda.empty_cache()

        source_pages = source_pages or []
        page_ref     = self._build_page_ref(source_pages)
        prompt_text  = self._build_prompt(query, context, page_ref)
        content = []
        if visual_pages:
            for vp in visual_pages[:3]:
                _, _, img = vp
                resized = img.copy()
                resized.thumbnail((384, 384))
                content.append({"type": "image", "image": resized})
        content.append({"type": "text", "text": prompt_text})

        messages   = [{"role": "user", "content": content}]
        text_input = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self.processor(
            text=[text_input],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(self.model.device)

        generated_ids = self.model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
        )
        trimmed = [
            out[len(inp):]
            for inp, out in zip(inputs.input_ids, generated_ids)
        ]
        answer = self.processor.batch_decode(
            trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]

        answer = self._strip_prompt_echo(answer)
        answer = self._clean_answer(answer)

        clean_check = re.sub(r"\[Source[^\]]*\]", "", answer, flags=re.IGNORECASE).strip()
        if not clean_check or len(clean_check.split()) < 4:
            return (
                f"The retrieved context does not contain a clear answer to this question. "
                f"[Source: {page_ref}]"
            )

        citation_pattern = re.compile(r"\[Source[s]?\s*:", re.IGNORECASE)
        if not citation_pattern.search(answer):
            answer = re.sub(r"\[Source[^\]]*\]?", "", answer, flags=re.IGNORECASE).strip()
            answer = f"{answer} [Source: {page_ref}]"

        return answer

Overwriting /content/src/generation.py


In [ ]:
%%writefile /content/src/evaluation.py

import re
import pandas as pd
from typing import List, Dict
from sentence_transformers import SentenceTransformer, util

semantic_eval_model = SentenceTransformer("all-MiniLM-L6-v2")

MAX_PAGES = 20

EVAL_QUERIES: List[Dict] = [
    {
        "query": "What is the central concept introduced in the World Bank 2024 report overview?",
        "type":  "text",
        "expected_keywords": ["middle", "income", "trap", "growth", "development"],
    },
    {
        "query": "What does the IMF say about inflation in advanced economies in 2024?",
        "type":  "text",
        "expected_keywords": ["inflation", "advanced", "economies", "disinflation", "percent"],
    },
    {
        "query": "What output growth figures for advanced economies appear in the IMF WEO projections table?",
        "type":  "table",
        "expected_keywords": ["advanced", "economies", "output", "growth", "projection"],
    },
    {
        "query": "What inflation rates for emerging market economies are shown in the IMF WEO overview table?",
        "type":  "table",
        "expected_keywords": ["inflation", "emerging", "market", "percent", "2024"],
    },
    {
        "query": "What does Figure 1.1 in the IMF WEO show about global growth or inflation trends?",
        "type":  "visual",
        "expected_keywords": ["figure", "global", "growth", "inflation", "trend"],
    },
    {
        "query": "What does Figure O.1 in the World Bank report show about middle-income countries?",
        "type":  "visual",
        "expected_keywords": ["figure", "middle", "income", "country", "growth"],
    },
    {
        "query": "What captions describe charts about economic risks in the IMF WEO opening chapter?",
        "type":  "image",
        "expected_keywords": ["figure", "risk", "outlook", "economic", "downside"],
    },
    {
        "query": "What are the captions of figures or boxes in the World Bank report overview pages?",
        "type":  "image",
        "expected_keywords": ["figure", "box", "overview", "income", "trap"],
    },
]

CITATION_RE = re.compile(r"\[Source[s]?\s*:[^\]]+\]", re.IGNORECASE)


def evaluate_system(queries, retriever, answer_gen, verbose=True) -> pd.DataFrame:
    results         = []
    page_violations = []

    for i, q in enumerate(queries):
        query       = q["query"]
        expected_kw = q["expected_keywords"]
        qtype       = q["type"]

        print(f"\n[{i+1}/{len(queries)}] ({qtype.upper()}) {query[:70]}...")

        retrieval    = retriever.retrieve(query)
        visual_pages = retrieval.get("visual_pages", [])
        source_pages = retrieval.get("source_pages", [])

        out_of_range = []
        for item in source_pages:
            page_num = item[0] if isinstance(item, tuple) else item
            if page_num > MAX_PAGES:
                out_of_range.append(page_num)
        if out_of_range:
            print(f"   PAGE RANGE WARNING: pages {out_of_range} > MAX_PAGES={MAX_PAGES}")
            page_violations.append({"query_idx": i + 1, "query": query, "bad_pages": out_of_range})
        else:
            print(f" Page range OK")

        answer = answer_gen.generate(
            query, retrieval["context"], visual_pages, source_pages=source_pages
        )

        ctx_lower    = retrieval["context"].lower()
        kw_ctx       = sum(1 for kw in expected_kw if kw.lower() in ctx_lower)
        ctx_kw_score = round(kw_ctx / max(1, len(expected_kw)), 2)

        ans_lower    = answer.lower()
        kw_ans       = sum(1 for kw in expected_kw if kw.lower() in ans_lower)
        ans_kw_score = round(kw_ans / max(1, len(expected_kw)), 2)

        expected_text  = " ".join(expected_kw)
        ans_emb        = semantic_eval_model.encode(answer,        convert_to_tensor=True)
        exp_emb        = semantic_eval_model.encode(expected_text, convert_to_tensor=True)
        semantic_score = float(max(0.0, min(1.0, util.cos_sim(ans_emb, exp_emb).item())))

        has_citation = bool(CITATION_RE.search(answer))

        if source_pages and isinstance(source_pages[0], tuple):
            pages_retrieved = [f"{s}:p{p}" for p, s in source_pages]
        else:
            pages_retrieved = [f"p{p}" for p in source_pages]

        modality_counts = retrieval.get("modality_counts", {})
        modalities_hit  = ", ".join(f"{k}:{v}" for k, v in modality_counts.items()) or "none"

        result = {
            "query":               query,
            "type":                qtype,
            "pages_retrieved":     pages_retrieved,
            "n_pages":             len(source_pages),
            "pages_in_range":      len(out_of_range) == 0,
            "modalities_hit":      modalities_hit,
            "context_kw_coverage": ctx_kw_score,
            "answer_kw_coverage":  ans_kw_score,
            "semantic_score":      round(semantic_score, 2),
            "has_citation":        has_citation,
            "answer_words":        len(answer.split()),
            "answer_preview":      answer[:250].replace("\n", " "),
        }
        results.append(result)

        if verbose:
            print(f"  Answer  : {answer[:200]}")
            print(f"  Pages   : {result['pages_retrieved']}")
            print(f"  Mods    : {modalities_hit}")
            print(f"  CtxKW:{ctx_kw_score:.0%} AnsKW:{ans_kw_score:.0%} Sem:{semantic_score:.0%} Cite:{has_citation}")

    df = pd.DataFrame(results)

    print(f"Queries evaluated        : {len(results)}")
    print(f"Pages-in-range rate      : {df['pages_in_range'].mean():.1%}")
    print(f"Avg context KW coverage  : {df['context_kw_coverage'].mean():.1%}")
    print(f"Avg answer KW coverage   : {df['answer_kw_coverage'].mean():.1%}")
    print(f"Avg semantic score       : {df['semantic_score'].mean():.1%}")
    print(f"Citation rate            : {df['has_citation'].mean():.1%}")
    print(f"Avg pages retrieved      : {df['n_pages'].mean():.1f}")
    print(f"Avg answer length (words): {df['answer_words'].mean():.0f}")

    if page_violations:
        print(f"\n {len(page_violations)} page-range violation(s):")
        for v in page_violations:
            print(f"   Query {v['query_idx']}: pages {v['bad_pages']} — {v['query'][:60]}")

    print("\nBy modality type:")
    print(
        df.groupby("type")[
            ["context_kw_coverage", "answer_kw_coverage", "semantic_score",
             "has_citation", "pages_in_range"]
        ].mean().round(2).to_string()
    )
    return df

Overwriting /content/src/evaluation.py


In [ ]:
import os

PDF_PATHS = [
    "/content/data/pdfs/world_bank_report.pdf",
    "/content/data/pdfs/imf_weo_report.pdf",
]
os.makedirs("/content/data/pdfs", exist_ok=True)

if not os.path.exists(PDF_PATHS[0]) or os.path.getsize(PDF_PATHS[0]) == 0:
    os.system(
        'wget -q -O /content/data/pdfs/world_bank_report.pdf '
        '"https://documents1.worldbank.org/curated/en/099080824150598470/pdf/'
        'P1807451cdbcc60881afdc19c40acb2e017.pdf"'
    )
print(f"World Bank: {os.path.getsize(PDF_PATHS[0])/1e6:.1f} MB")

if not os.path.exists(PDF_PATHS[1]) or os.path.getsize(PDF_PATHS[1]) == 0:
    os.system(
        'wget -q -O /content/data/pdfs/imf_weo_report.pdf '
        '"https://www.imf.org/-/media/files/publications/weo/2024/april/english/text.pdf"'
    )
print(f"IMF WEO: {os.path.getsize(PDF_PATHS[1])/1e6:.1f} MB")

PDF_PATHS = [p for p in PDF_PATHS if os.path.exists(p) and os.path.getsize(p) > 1000]
print(f"PDFs to process: {PDF_PATHS}")

World Bank: 2.1 MB
IMF WEO: 9.9 MB
PDFs to process: ['/content/data/pdfs/world_bank_report.pdf', '/content/data/pdfs/imf_weo_report.pdf']


In [ ]:
import sys
for mod in ["ingestion", "indexing", "retrieval", "generation", "evaluation"]:
    sys.modules.pop(mod, None)
sys.path.insert(0, "/content/src")

from ingestion import MultiModalIngestionPipeline

pipeline = MultiModalIngestionPipeline(dpi=150)
all_chunks, page_images = pipeline.ingest(PDF_PATHS)
print(f"\nTotal chunks : {len(all_chunks)}")
print(f"Total pages  : {len(page_images)}")


 world_bank_report: 48 pages total — processing first 20


  world_bank_report: 100%|██████████| 20/20 [00:13<00:00,  1.47it/s]



 imf_weo_report: 202 pages total — processing first 20


  imf_weo_report: 100%|██████████| 20/20 [00:11<00:00,  1.73it/s]


 Ingestion complete — 51 chunks from 2 PDFs
   image_caption: 10
   table: 4
   text: 37
   Page images: 40

Total chunks : 51
Total pages  : 40


In [ ]:
from indexing import ColPaliIndexer, TextIndexer
from retrieval import DualModeRetriever

colpali_indexer = ColPaliIndexer()
colpali_indexer.embed_pages(page_images, batch_size=1)

text_indexer = TextIndexer("all-MiniLM-L6-v2")
text_indexer.build_index(all_chunks)

retriever = DualModeRetriever(colpali_indexer, text_indexer, all_chunks, page_images)
print(" DualModeRetriever ready")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
`torch_dtype` is deprecated! Use `dtype` instead!


Loading ColPali: vidore/colpali-v1.3-hf


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


ColPali loaded (4-bit quantized)

 Embedding 40 pages with ColPali...


ColPali pages: 100%|██████████| 40/40 [01:41<00:00,  2.55s/it]


 40 page embeddings stored
Loading text embedder: all-MiniLM-L6-v2
Text embedder loaded (dim=384)

 Embedding 51 chunks with MiniLM...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index built: 51 vectors
 DualModeRetriever ready


In [ ]:
from generation import AnswerGenerator
answer_gen = AnswerGenerator()

Loading VLM: Qwen/Qwen2-VL-2B-Instruct


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 Qwen2-VL-2B loaded (4-bit, max_new_tokens=450)


In [ ]:
import sys
for mod in ["ingestion", "indexing", "retrieval", "generation", "evaluation"]:
    sys.modules.pop(mod, None)

from retrieval import DualModeRetriever

retriever = DualModeRetriever(colpali_indexer, text_indexer, all_chunks, page_images)

ret = retriever.retrieve("What is the middle income trap?")
print(" retrieve() works:", list(ret.keys()))

 retrieve() works: ['context', 'source_pages', 'retrieved_chunks', 'visual_pages', 'modality_counts']


In [ ]:
def _fmt_pages(source_pages):
    out = []
    for item in source_pages:
        if isinstance(item, tuple):
            page_num, source_doc = item
            out.append(f"{source_doc}:p{page_num}")
        else:
            out.append(f"p{item}")
    return out

smoke_queries = [
    "What is the global economic growth projection for 2025?",
    "What is the middle income trap?",
    "What inflation rates are shown in the IMF WEO table for emerging markets?",
    "What does Figure 1.1 in the IMF WEO show?",
]

print("SMOKE TESTS")
for q in smoke_queries:
    ret = retriever.retrieve(q)
    ans = answer_gen.generate(q, ret["context"], ret["visual_pages"], source_pages=ret["source_pages"])
    print(f"\nQ : {q}")
    print(f"A : {ans}")
    print(f"   pages={_fmt_pages(ret['source_pages'])}  modalities={ret['modality_counts']}")

SMOKE TESTS


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Q : What is the global economic growth projection for 2025?
A : Based on the information provided, the global economic growth projection for 2025 is: 1. According to the World Bank report, middle-income countries are expected to move towards higher-income status over time. 2. The report states that countries must engineer two successive transitions to move towards higher-income status. 3. The transition process involves increasing investment, infusing innovation, and producing productivity. 4. The key factors influencing this transition include maintaining high levels of investment, adding infusions to accelerate investments, and then augmenting the mix with innovation policies. So, the global economic growth projection for 2025 is likely to be towards higher-income status for middle-income countries. [Source: p.19 (world_bank_report), p.16 (world_bank_report), p.18 (world_bank_report), p.15 (imf_weo_report)]
   pages=['world_bank_report:p19', 'world_bank_report:p16', 'world_bank_repo

In [ ]:
from evaluation import evaluate_system, EVAL_QUERIES

eval_df = evaluate_system(EVAL_QUERIES, retriever, answer_gen, verbose=True)

try:
    from IPython.display import display
    display(eval_df[[
        "query", "type", "n_pages", "modalities_hit",
        "context_kw_coverage", "answer_kw_coverage", "semantic_score",
        "has_citation", "answer_words", "answer_preview",
    ]])
except ImportError:
    print(eval_df.to_string())


[1/8] (TEXT) What is the central concept introduced in the World Bank 2024 report o...
 Page range OK
  Answer  : The central concept introduced in the World Bank 2024 report overview is the "middle-income trap." [Source: p.3 (world_bank_report), p.1 (world_bank_report), p.4 (world_bank_report), p.9 (world_bank_r
  Pages   : ['world_bank_report:p3', 'world_bank_report:p1', 'world_bank_report:p4', 'world_bank_report:p9', 'world_bank_report:p5']
  Mods    : text:4
  CtxKW:100% AnsKW:60% Sem:62% Cite:True

[2/8] (TEXT) What does the IMF say about inflation in advanced economies in 2024?...
 Page range OK
  Answer  : The IMF says that global headline inflation is expected to fall from an annual average of 6.8 percent in 2023 to 5.9 percent in 2024 and 4.5 percent in 2025, with advanced economies returning to their
  Pages   : ['imf_weo_report:p18', 'imf_weo_report:p15', 'imf_weo_report:p7', 'imf_weo_report:p8', 'imf_weo_report:p9']
  Mods    : text:4, image_caption:5
  CtxKW:100% AnsKW:80

,query,type,n_pages,modalities_hit,context_kw_coverage,answer_kw_coverage,semantic_score,has_citation,answer_words,answer_preview
0,What is the central concept introduced in the ...,text,5,text:4,1.0,0.6,0.62,True,24,The central concept introduced in the World Ba...
1,What does the IMF say about inflation in advan...,text,5,"text:4, image_caption:5",1.0,0.8,0.58,True,54,The IMF says that global headline inflation is...
2,What output growth figures for advanced econom...,table,5,"table:2, text:4, image_caption:4",1.0,1.0,0.73,True,24,The IMF WEO projections table does not provide...
3,What inflation rates for emerging market econo...,table,5,"image_caption:5, text:4",1.0,0.6,0.55,True,57,The inflation rates for emerging market econom...
4,What does Figure 1.1 in the IMF WEO show about...,visual,5,"table:2, text:4, image_caption:4",1.0,1.0,0.75,True,51,Figure 1.1 shows the trend of global growth an...
5,What does Figure O.1 in the World Bank report ...,visual,5,"table:4, text:4",1.0,0.6,0.69,True,30,Figure O.1 shows that the share of middle-inco...
6,What captions describe charts about economic r...,image,5,"text:4, image_caption:2",0.6,0.0,0.38,True,19,Box 4.1. Industrial Policies in Emerging Marke...
7,What are the captions of figures or boxes in t...,image,5,text:4,0.8,0.6,0.55,True,24,The caption of figures or boxes in the World B...


In [ ]:
import gradio as gr


def _fmt_source_pages_str(source_pages: list) -> str:
    parts = []
    for item in source_pages:
        if isinstance(item, tuple):
            page_num, source_doc = item
            parts.append(f"{source_doc} p.{page_num}")
        else:
            parts.append(f"p.{item}")
    return ", ".join(parts) if parts else "unknown"


def chat_fn(query: str, history: list):
    if not query.strip():
        return "", history
    try:
        ret = retriever.retrieve(query)
        source_pages = ret.get("source_pages", [])
        visual_pages = ret.get("visual_pages",  [])

        answer = answer_gen.generate(
            query, ret["context"], visual_pages, source_pages=source_pages,
        )

        pages_str = _fmt_source_pages_str(source_pages)
        mod_str   = (
            " | ".join(f"{k}:{v}" for k, v in ret.get("modality_counts", {}).items())
            or "none"
        )
        response = f"{answer}\n\n📍 **Sources:** {pages_str}\n🔍 **Modalities:** {mod_str}"
        history.append((query, response))
        return "", history

    except Exception as exc:
        import traceback
        traceback.print_exc()
        history.append((query, f"⚠️ Error: {exc}"))
        return "", history


with gr.Blocks(title="Multi-Modal RAG", theme=gr.themes.Base(primary_hue="blue")) as demo:
    gr.Markdown("# 📚 Multi-Modal RAG System")
    chatbot   = gr.Chatbot(label="Conversation", height=520, bubble_full_width=False)
    query_box = gr.Textbox(placeholder="Ask a question about the documents...", label="Your question", lines=2)

    with gr.Row():
        submit_btn = gr.Button("Ask ➤",   variant="primary")
        clear_btn  = gr.Button("🗑 Clear", variant="secondary")

    for trigger in [submit_btn.click, query_box.submit]:
        trigger(fn=chat_fn, inputs=[query_box, chatbot], outputs=[query_box, chatbot], api_name=False)

    clear_btn.click(fn=lambda: ([], ""), outputs=[chatbot, query_box], api_name=False)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://7ac49bcdc846d76399.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
